To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

Read our **[Qwen3 Guide](https://docs.unsloth.ai/basics/qwen3-how-to-run-and-fine-tune)** and check out our new **[Dynamic 2.0](https://docs.unsloth.ai/basics/unsloth-dynamic-2.0-ggufs)** quants which outperforms other quantization methods!

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.4.7: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.4.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Alpaca.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [5]:
# @title 4. Definir y Preparar el Dataset Custom (Adaptando el Script que me diste)
# Define tu dataset directamente en el código como una lista de diccionarios.
# Cada diccionario debe tener las claves 'prompt' y 'response'.

dataset_json = [
  {
    "prompt": "¿Los pedidos por que compañía los despachan?",
    "response": "Hacemos envíos gratis a todo el país por Correo Argentino o Andreani, para que te llegue rápido y seguro. Además, si elegís Andreani, suele ser más rápido según tu zona. ¿Te gustaría que te ayude a realizar un pedido? Podés ver todos nuestros productos y hacer la compra desde nuestra página web https://www.modeon.com.ar/ ¡Te espero! ✨"
  },
  {
    "prompt": "Hola. Quería consultar por la cafetera portátil.\n\n¿Funciona con cápsulas reutilizables?\nO sin capsulas?",
    "response": "Hola como estas?\nLa cafetera puede usar capsulas chicas (Nespresso), capsulas grandes (Dolce Gusto) y tambien cafe molido directamente ! Ademas, puede calentar el agua si le pones agua fria o podes ponerle agua caliente directamente\nhttps://www.modeon.com.ar/cafetera-portatil/\nComo funciona con cafe molido directamente te recomendaria hacer eso en vez de usar capsula reutilizable\nPero poder se podria, solo que no tendria mucho sentido!"
  },
  {
    "prompt": "Viene con todo para usar café molido?",
    "response": "Sii! Es un compartimento muy sencillo\nY viene con instrucciones en Español"
  },
  {
    "prompt": "Buenas noches. Traen hasta Pinamar?\n¿A qué precio está la cafetera portátil por transferencia?",
    "response": "Holaa Gisela como estas? Tenemos envíos gratis a todo el pais, incluido Pinamar\nhttps://www.modeon.com.ar/cafetera-portatil/\nTe quedaria por transferencia a $131.600\nCualquier duda podes preguntarle! ✨"
  },
  {
    "prompt": "Hola tienen algun lugar para retirar?",
    "response": "Hola!!\nSolo vendemos online\nEn que producto estas interesado?"
  },
  {
    "prompt": "En el que sirve para dormir",
    "response": "https://www.modeon.com.ar/dispositivos-dormir/\nAhi te comparto los modelos\nFuncionan con la terapia CES\nLos modelos difieren en el tipo de agarre y lo estético mas que nada"
  },
  {
    "prompt": "Hola buenos días!\nQue calidad de imagen tiene la consola portátil R36S?",
    "response": "Hola, buenos dias!\nLa calidad es buenisima la verdad! Se ve hasta mejor de nitido de lo que se puede apreciar en los videos\nAhi te digo que resolución tiene\nLa resolucion es de 640x480, espectacular para la pantalla que es de 3,5 pulgadas"
  },
  {
    "prompt": "Error 404",
    "response": "Ahi te paso el link indicado\nGracias por avisar! A veces hay errores cuando se muestran los linls en facebook o insta\nhttps://www.modeon.com.ar/productos/R36S128GB/"
  },
  {
    "prompt": "La transferencia va para MODEON.GAL?\nAldana Micaela Stornini",
    "response": "Es correcto !\nEstos son los datos para la transferencia:\n------\nBanco: Galicia\nAlias: MODEON.GAL\nTitular: Aldana Stornini\nCBU: 0070221030004032738705\n------"
  },
  {
    "prompt": "No no voy a comprar\nIgual gracias por la atención",
    "response": "Paso algo que te hizo cambiar de opinión? Nos serviria mucho el feedback"
  },
  {
    "prompt": "La encontré en menor precio en una tienda de mercado libre",
    "response": "No tiene ni funda (que si la queres comorar aparte esta alrededor de 25-30 mil pesos) y tiene la mitad de gb ( o sea la mitad de juegos que la nuestra)\nY dudo que tengan garantia si te llega a pasar algo despies de los 30 dias de tu compra"
  },
  {
    "prompt": "Cancelo la compra entonces?",
    "response": "Es tu decision!"
  },
  {
    "prompt": "Ya la cancelé\nNo tuve en cuenta la funda que ofrecen ustedes\nQuiero la consola negra transparente\nSi les quedan\nSi no quedan entonces quiero la transparente violeta\nEn cuánto tiempo me estaría llegando?",
    "response": "Dale,  tarda 3/4 dias habiles el envío!\nAhi reabro la orden que la habia cancelado"
  },
  {
    "prompt": "¿Cuántas horas hay que cargarla?\nAsí no arruino la batería jaja\nPorque busqué en Google y no me aparece nada concreto",
    "response": "3-4hs esta bien, con el agujerito de la izquierda\ny con un enchufe normal (no de carga rapida)\neso nos recomendó el fabricante"
  },
  {
    "prompt": "Buenas noches",
    "response": "Hola Sol como estas? En que podemos ayudsrte\nAyudarte*"
  },
  {
    "prompt": "Hola,perdón la molestia quería hacer unas consultas por un producto\nNo especifica que accesorios trae\nY si se puede.pagar con tarjeta credito o solo por mercado pago?",
    "response": "No es molestia! Hace todas las consultas\n<adjunto: 00000011-PHOTO-2025-05-01-22-17-19.jpg>\nY todo lo del medio\nCon los adaptadores podes cambiar el tipo de capsula. La cafetera puede usar capsulas chicas (Nespresso), capsulas grandes (Dolce Gusto) y tambien cafe molido directamente ! Ademas, puede calentar el agua si le pones agua fria o podes ponerle agua caliente directamente\nEn nuestra pagina ofrecemos envío gratis a todo el pais\nY la posibilidad de pagar con todas las tarjetas\nSi es tarjeta de credito tenes hasta 6 cuotas sin interes\nY si elegis por transferencia tenes 20% de descuento!\nHaciendo tu compra hoy la preparamos para despachar mañana !"
  },
  {
    "prompt": "Genial, muchas gracias.\nOsea llegaria mañana viernes 02?",
    "response": "Nono\nDespachamos mañana nosotros\nO sea la preparamos y la dejamos en sucursal de Correo Argentino o Andreani\nNuestra sucursal\nY despues empieza a viajar\nPodes elegir Correo Argentino o Andreani en el momento de la compra, a domicilio o sucursal\nO sea que como mañana es viernes, llegaria para la semana que viene!\nDespachar signfiica que dejamos el paquete para que la empresa transportista lo lleve\nClaro, es verdad perdon."
  },
  {
    "prompt": "Me voy a comprarla\nFeliz dia del trabajador ☺️",
    "response": "Gracias!! Feliz dia para vos tambien\nGenial! Cualquier cosa avisame"
  },
  {
    "prompt": "Buenas tardes. Ya compre el producto",
    "response": "Sos Maria Sol Gomez?\nAhi lo estamos preparando ! Gracias x confirmar\nHoy te estará llegando un email con el seguimiento"
  },
  {
    "prompt": "Hola buenas",
    "response": "Hola Marias\nMatias\nComo estas?\nEn que podemos ayudarte?"
  },
  {
    "prompt": "Si ví que tenían una consola\nPortátil",
    "response": "Te comparto el link\nhttps://www.modeon.com.ar/productos/R36S128GB/\nViene con estuche de regalo! Y tenemos envios gratis a todo el pais"
  },
  {
    "prompt": "Se puede abonar en cuotas?",
    "response": "Siii obvio! Actualmente tenemos 6 cuotas sin interes con cualquier banco ✨✨✨"
  },
  {
    "prompt": "¡Hola! Podrías darme más información de...\nEntrega solo x correo????",
    "response": "Holaa! Buenas tardes!\nSiii, correo argentino o andreani"
  },
  {
    "prompt": "Holaaaaaa\nSi… lo voy a comprar\nMañana",
    "response": "Gracias por tu respuesta! La guardamos hasta mañana entonces. El correo va a estar abierto asi que lo despachamos mañana mismo\n✨✨\nPerfecto"
  },
  {
    "prompt": "No me permite hacer transferencia",
    "response": "Estos son los datos para la transferencia:\n——\nBanco: Galicia\nAlias: MODEON.GAL\nTitular: Aldana Stornini\nCBU: 0070221030004032738705\n——"
  },
  {
    "prompt": "Buen día\nSi, la verdad que necesito que llegue esta semana por favor",
    "response": "No te lo puedo garantizar, por que hay feriados\nNo te va a llegar seguramente\nCancelo o la enviamos igual?"
  },
  {
    "prompt": "Cancelar la compra o el envío?",
    "response": "cancelo tu compra y se te devuelve el dinero\no hacemos el envio hoy igual y te llegara la semanaa que viene?"
  },
  {
    "prompt": "Y si compro la semana que viene es lo mismo?\nO sea es una semana de espera siempre\n?\nPorque yo lo necesito para el viernes\nDe esta semana",
    "response": "No es asi\nUna vez que vos haces la compra, nosotros nos comprometemos a enviar lo antes posible\ncomo la hiciste ayer a las 22hs\nhoy podemos despachar en Andreani\nhoy es masrtes\nhoy y mañana trabajan\npero Jueves y viernes no porque es feriado\nlo mas probable es que quede el paquete en Andreani y te lo entreguen la semana que viene, lunes o martes\ncuando vuelvan a trabajar\nno depende de nosotros"
  },
  {
    "prompt": "A qué hora despachan ?\nPor la mañana lo llevarían a Andreani?",
    "response": "a las 17hs hoy"
  },
  {
    "prompt": "Ok, entonces llega la semana que viene si o si",
    "response": "con 90% de probabilidad si, por los feriados"
  },
  {
    "prompt": "Dame 15 minutos y te contesto\nPorque encima el lunes no estoy",
    "response": "Si queres pongo a Andreani Sucursal\nla que te quede mas cerca"
  },
  {
    "prompt": "Cancela la compra por favor",
    "response": "Dale\n<adjunto: 00000045-PHOTO-2025-04-29-09-48-31.jpg>\nlos $1245,27 son costos que incurrimos nosotros, porque se cobran impuestos aunque cancelemos\n???\ndije que incurrimos nosotros\na vos se te devuelve la totalidad\nnosotros perdimos"
  },
  {
    "prompt": "O sea que igual perdí\nOk\nIgual gracias por el contacto y la gestión\nSi. Pero el tema es que quería la cafetera, una pena que no me di cuenta lo de los días\nNo me refería al costo sino a no tener el procuro\nClaro, pero igual llega la semana que viene\nLa idea era usarlo esta semana\nLo sé, por eso dije una pena que no me di cuenta lo de los días",
    "response": "Podiamos ir que vaya a sucursal\npodiamos hace que vaya a sucursal*\ny la retirabas cuando este disponible\nno podemos hacer nada con los feriados y fin de semana\nBueno! Esperamos tu compra si te arrepentis mas adelante\ny queres volver a hacerla\nCuando leí que decía 2/5 me cerró enseguida\nSi no consigo otra que me llegue antes, seguro que se la compro a ustedes\nGracias\nEl cupón sigue sirviendo?\nEl de primera compra"
  },
  {
    "prompt": "Ok, muchas gracias\nConsulta\nSi decido comprarla nuevamente hoy, hasta qué hora tengo tiempo de hacerlo como para que ustedes lo puedan despachar hoy mismo",
    "response": "Eso no estoy seguro si te va a dejar activalro! De ultima avisanos y te transferimos 10mil si no te deja ponerlo\nHoy antes de las 12hs!"
  },
  {
    "prompt": "Hola. Buen día. Hice una simulación para ver costos y si mandaban a mi CP. Por ahora doy de baja la compra. Seguramente mas adelante la haré. Gracias",
    "response": "Dale no hay problema\nHay un 20% adicional de descuento eligiendo por transferencia\nEsperamos tu compra !\n👍\nPor ahora la oferta incluye funda por tiempo limitado, te aviso por las dudas"
  }
]

# Importa la biblioteca datasets para trabajar con DataSets de Hugging Face
from datasets import Dataset

# --- Inicio de la Adaptación del Script que me pasaste ---

# 1. Define el prompt template que me pasaste
# Este template tiene espacios para Instruction, Input y Response
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Necesitas el tokenizador para obtener el EOS_TOKEN.
# El tokenizador se cargará en la Celda 5.
# Asegúrate de que 'tokenizer' esté definido antes de ejecutar la función de formateo.
# Si ejecutas las celdas en orden, 'tokenizer' estará disponible.
# EOS_TOKEN = tokenizer.eos_token # Esto se hará dentro de la función para asegurar que el tokenizer esté cargado

# 2. Adapta la función de formateo para leer tus claves 'prompt' y 'response'
# y colocarlas en el template alpaca_prompt.
def formatting_prompts_func_adapted(examples):
    # **AQUÍ SE ADAPTA**: Leemos tus claves 'prompt' y 'response'
    prompts = examples["prompt"]
    responses = examples["response"]

    texts = []

    # Asegúrate de que 'tokenizer' esté accesible en este punto
    if 'tokenizer' not in globals() or tokenizer.eos_token is None:
         print("Error: Tokenizer no definido o EOS_TOKEN no disponible. Asegúrate de ejecutar la Celda 5 antes.")
         # Retorna un indicador de error para evitar que el .map falle completamente
         # Dependiendo del dataset.map, un error aquí puede detener el proceso.
         # Para datasets pequeños y batched=True, esto podría funcionar.
         return {"text": [None] * len(prompts)} # Devuelve Nones para mantener el batch size

    EOS_TOKEN = tokenizer.eos_token # Obtén el EOS_TOKEN del tokenizador cargado

    # Iteramos sobre los prompts y respuestas en el batch
    for prompt, response in zip(prompts, responses):
        # Formatea usando el alpaca_prompt template:
        # instruction = prompt
        # input = "" (dejamos el input vacío ya que no tienes esa clave separada)
        # response = response
        # Añadimos el EOS_TOKEN al final
        if prompt is not None and response is not None: # Asegurarse de que no son None si hay errores
             text = alpaca_prompt.format(prompt, "", response) + EOS_TOKEN
             texts.append(text)
        else:
             texts.append(None) # Añadir None si el ejemplo fue inválido

    return { "text" : texts, } # Devolvemos un diccionario con la lista de textos formateados


# 3. En lugar de load_dataset("yahma/alpaca-cleaned", ...), carga tu JSON
# Convertimos tu lista de diccionarios a un Dataset de Hugging Face.
# Este Dataset inicialmente tendrá las columnas 'prompt' y 'response'.
raw_hf_dataset = Dataset.from_list(dataset_json)

print("Dataset custom JSON cargado como Hugging Face Dataset.")
print(f"Columnas iniciales: {raw_hf_dataset.column_names}")

# 4. Aplica la función de formateo ADAPTADA a tu Dataset.
# Esto creará una nueva columna 'text' en el Dataset, con los ejemplos formateados
# usando el alpaca_prompt template y tus datos.
# Usa batched=True para procesar ejemplos en batches, lo cual es más eficiente.
# remove_columns=raw_hf_dataset.column_names elimina las columnas originales ('prompt', 'response')
# para ahorrar memoria, dejando solo la columna 'text'.

processed_hf_dataset = raw_hf_dataset.map(
    formatting_prompts_func_adapted,
    batched=True,
    remove_columns=raw_hf_dataset.column_names # Eliminamos las columnas originales
)

# Filtra los ejemplos que pudieron haber resultado en None si hubo algún error en el formateo
processed_hf_dataset = processed_hf_dataset.filter(lambda example: example["text"] is not None)


# --- Fin de la Adaptación del Script que me pasaste ---


# El Dataset listo para entrenar es 'processed_hf_dataset'
# Ahora este dataset tiene una columna 'text' con tus ejemplos formateados así:
# "Below is an instruction...\n### Instruction:\n¿Tu Pregunta?\n\n### Input:\n\n### Response:\nTu Respuesta!</s>"

print("\nDataset custom formateado para entrenamiento (primer ejemplo):")
# Muestra el contenido del campo 'text' del primer ejemplo formateado
if len(processed_hf_dataset) > 0:
  print(processed_hf_dataset[0]['text'])
  print(f"\nTotal de ejemplos formateados listos para usar: {len(processed_hf_dataset)}")
else:
  print("El dataset formateado está vacío. Revisa tus datos y la función de formateo.")

Dataset custom JSON cargado como Hugging Face Dataset.
Columnas iniciales: ['prompt', 'response']


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Filter:   0%|          | 0/35 [00:00<?, ? examples/s]


Dataset custom formateado para entrenamiento (primer ejemplo):
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
¿Los pedidos por que compañía los despachan?

### Input:


### Response:
Hacemos envíos gratis a todo el país por Correo Argentino o Andreani, para que te llegue rápido y seguro. Además, si elegís Andreani, suele ser más rápido según tu zona. ¿Te gustaría que te ayude a realizar un pedido? Podés ver todos nuestros productos y hacer la compra desde nuestra página web https://www.modeon.com.ar/ ¡Te espero! ✨<|end_of_text|>

Total de ejemplos formateados listos para usar: 35


<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = processed_hf_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/35 [00:00<?, ? examples/s]

In [8]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
7.135 GB of memory reserved.


In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 35 | Num Epochs = 15 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.468300
2,3.595100
3,3.269600
4,3.258100
5,3.205700
6,2.688700
7,2.405600
8,2.439100
9,2.132400
10,2.404200


In [10]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

264.8718 seconds used for training.
4.41 minutes used for training.
Peak reserved memory = 7.135 GB.
Peak reserved memory for training = 0.0 GB.
Peak reserved memory % of max memory = 48.402 %.
Peak reserved memory for training % of max memory = 0.0 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

**[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct [here](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Unsloth_Studio.ipynb)**

In [11]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Responde los mensajes, trabajas en un emprendimiento de venta de informatica.", # instruction
        "hola, hacen envios a cordoba?", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nResponde los mensajes, trabajas en un emprendimiento de venta de informatica.\n\n### Input:\nhola, hacen envios a cordoba?\n\n### Response:\nHolaa! Tenemos envíos gratis a todo el pais, incluido Cordoba\nhttps://www.modeon.com.ar/transparente/\nTe quedaria por la mañana hoy!<|end_of_text|>']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Continue the fibonnaci sequence.

### Input:
1, 1, 2, 3, 5, 8

### Response:
13, 21, 34, 55, 89, 144<|end_of_text|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [13]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [14]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is a famous tall tower in Paris?

### Input:


### Response:
La Torre Eiffel es una torre de 324 metros de altura, construida en 1889 para la Exposición Universal de París. Actualmente es un icono de la ciudad y una de las atracciones mas visitadas de Francia.<|end_of_text|>


You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [15]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [16]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
